In [4]:
import json
import subprocess
from pathlib import Path

import dcm2niix
import pandas as pd

In [5]:
# ==========================================
# 1. CONFIGURATION & DIRECTORY SETUP
# ==========================================
MASTER_CSV = "../../results/dataset_creation/clean_dedup_fmri_to_t1_master.csv"
MANIFEST_CSV = "../../results/dataset_creation/bids_conversion_manifest.csv"
FMRI_ROOT = Path("../../data/raw_fmri/ADNI")
SMRI_ROOT = Path("../../data/raw_smri/ADNI")
BIDS_ROOT = Path("../../data/raw_bids")

In [6]:
BIDS_ROOT.mkdir(parents=True, exist_ok=True)

# Create required BIDS root dataset_description.json
dataset_desc_path = BIDS_ROOT / "dataset_description.json"
if not dataset_desc_path.exists():
    dataset_desc = {
        "Name": "ADNI Resting-State fMRI and T1w BIDS Dataset",
        "BIDSVersion": "1.10.1",
        "DatasetType": "raw",
    }
    dataset_desc_path.write_text(json.dumps(dataset_desc, indent=2), encoding="utf-8")

master_df = pd.read_csv(MASTER_CSV)
print(f"Loaded master table with {len(master_df)} sessions.")

Loaded master table with 3278 sessions.


In [7]:
# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def sanitize_id(subj_str):
    """Converts ADNI ID '002_S_0413' -> 'sub-002S0413'"""
    return "sub-" + str(subj_str).replace("_", "")


def sanitize_date_ses(date_str):
    """Converts '7/27/2012' or '2012-07-27' -> 'ses-20120727'"""
    dt = pd.to_datetime(date_str, format="mixed")
    return "ses-" + dt.strftime("%Y%m%d")


def convert_dicom_to_bids(
    src_dcm_dir: Path, out_dir: Path, out_stem: str, is_bold: bool = False
):
    """
    Converts a DICOM folder directly to BIDS NIfTI (.nii.gz) + sidecar (.json) via dcm2niix.
    Skips conversion if target files already exist (resumable).
    Returns (success: bool, status_msg: str).
    """
    nii_path = out_dir / f"{out_stem}.nii.gz"
    json_path = out_dir / f"{out_stem}.json"

    if nii_path.exists() and json_path.exists():
        return True, "already_exists"

    out_dir.mkdir(parents=True, exist_ok=True)
    # Run dcm2niix with cwd=src_dcm_dir and input="." to avoid Windows 260-char MAX_PATH limits
    cmd = [
        str(dcm2niix.bin),
        "-b",
        "y",  # generate BIDS JSON sidecar
        "-z",
        "y",  # gzip compress to .nii.gz
        "-f",
        out_stem,  # exact BIDS filename stem
        "-o",
        str(out_dir.resolve()),  # resolved output directory
        ".",  # current directory (src_dcm_dir)
    ]
    res = subprocess.run(
        cmd, cwd=str(src_dcm_dir.resolve()), capture_output=True, text=True
    )

    if not nii_path.exists() or not json_path.exists():
        # Check if dcm2niix appended a suffix (e.g., _e1, _a)
        produced_nii = sorted(out_dir.glob(f"{out_stem}*.nii.gz"))
        produced_json = sorted(out_dir.glob(f"{out_stem}*.json"))
        if len(produced_nii) == 1 and len(produced_json) == 1:
            produced_nii[0].rename(nii_path)
            produced_json[0].rename(json_path)
        else:
            err_lines = [
                l.strip()
                for l in (res.stderr + "\n" + res.stdout).splitlines()
                if "Error" in l or "missing" in l
            ]
            err_hint = err_lines[0][:120] if err_lines else ""
            return (
                False,
                f"dcm2niix_failed (rc={res.returncode}, nii={len(produced_nii)}) {err_hint}".strip(),
            )

    # Ensure required BIDS 'TaskName' field is present in BOLD JSON sidecar
    if is_bold and json_path.exists():
        meta = json.loads(json_path.read_text(encoding="utf-8"))
        if "TaskName" not in meta:
            meta["TaskName"] = "rest"
            json_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    return True, "converted"

In [ ]:
# x_df = master_df[:5]
# x_df

,Image Data ID,Subject,Group,Sex,Age,Visit,Modality,Description,Acq Date,raw_id,...,SlicesPerVolume,SliceThickness,LONIStudy,ReceiveCoilName,T1_Image_ID,T1_Raw_ID,T1_Study_Date,T1_Description,Same_LONIStudy,Days_Apart
0,I238623,002_S_0295,CN,M,90,v06,fMRI,Resting State fMRI,6/02/2011,238623,...,48.0,3.31,34310.0,SENSE-Head-8,I238627,238627,2011-06-02,MPRAGE,True,0
1,I303069,002_S_0295,CN,M,91,v11,fMRI,Resting State fMRI,5/10/2012,303069,...,48.0,3.31,45566.0,SENSE-Head-8,I303066,303066,2012-05-10,MPRAGE,True,0
2,I240811,002_S_0413,CN,F,82,v06,fMRI,Resting State fMRI,6/16/2011,240811,...,48.0,3.31,34751.0,SENSE-Head-8,I240812,240812,2011-06-16,MPRAGE,True,0
3,I304790,002_S_0413,CN,F,82,v11,fMRI,Resting State fMRI,5/15/2012,304790,...,48.0,3.31,45738.0,SENSE-Head-8,I304793,304793,2012-05-15,MPRAGE,True,0
4,I371994,002_S_0413,CN,F,83,v21,fMRI,Resting State fMRI,5/10/2013,371994,...,48.0,3.31,58370.0,SENSE-Head-8,I371991,371991,2013-05-10,MPRAGE,True,0


In [ ]:
# # Pilot test on the first 5 sessions (x_df)
# for idx, row in x_df.iterrows():
#     subject_raw = row["Subject"]
#     fmri_id = str(row["Image Data ID"])
#     t1_id = str(row["T1_Image_ID"])

#     sub_id = sanitize_id(subject_raw)
#     ses_id = sanitize_date_ses(row["Acq Date"])

#     ses_dir = BIDS_ROOT / sub_id / ses_id
#     anat_dir = ses_dir / "anat"
#     func_dir = ses_dir / "func"

#     fmri_src = list((FMRI_ROOT / subject_raw).glob(f"*/*/{fmri_id}"))
#     t1_src = list((SMRI_ROOT / subject_raw).glob(f"*/*/{t1_id}"))

#     bold_stem = f"{sub_id}_{ses_id}_task-rest_bold"
#     t1_stem = f"{sub_id}_{ses_id}_T1w"

#     ok_f, msg_f = convert_dicom_to_bids(fmri_src[0], func_dir, bold_stem, is_bold=True)
#     ok_t, msg_t = convert_dicom_to_bids(t1_src[0], anat_dir, t1_stem, is_bold=False)
#     print(f"{sub_id} {ses_id} | BOLD: {msg_f} | T1w: {msg_t}")


sub-002S0295 ses-20110602 | BOLD: converted | T1w: converted
sub-002S0295 ses-20120510 | BOLD: converted | T1w: converted
sub-002S0413 ses-20110616 | BOLD: converted | T1w: converted
sub-002S0413 ses-20120515 | BOLD: converted | T1w: converted
sub-002S0413 ses-20130510 | BOLD: converted | T1w: converted


In [ ]:
# ==========================================
# 3. FULL BIDS CONVERSION LOOP (dcm2niix)
# ==========================================
manifest = []
missing_fmri = 0
missing_smri = 0
total = len(master_df)

for idx, row in master_df.iterrows():
    subject_raw = row["Subject"]  # e.g., '002_S_0413'
    fmri_id = str(row["Image Data ID"])  # e.g., 'I322442'
    t1_id = str(row["T1_Image_ID"])  # e.g., 'I322438'

    fmri_date = row["Acq Date"]
    t1_date = row["T1_Study_Date"]

    sub_id = sanitize_id(subject_raw)
    ses_id = sanitize_date_ses(fmri_date)

    ses_dir = BIDS_ROOT / sub_id / ses_id
    anat_dir = ses_dir / "anat"
    func_dir = ses_dir / "func"

    # --- A. LOCATE & CONVERT fMRI DICOMS ---
    fmri_src_folders = list((FMRI_ROOT / subject_raw).glob(f"*/*/{fmri_id}"))
    if fmri_src_folders:
        bold_stem = f"{sub_id}_{ses_id}_task-rest_bold"
        fmri_success, fmri_status = convert_dicom_to_bids(
            fmri_src_folders[0], func_dir, bold_stem, is_bold=True
        )
    else:
        fmri_success, fmri_status = False, "missing_source_folder"
        missing_fmri += 1

    # --- B. LOCATE & CONVERT sMRI (T1w) DICOMS ---
    t1_src_folders = list((SMRI_ROOT / subject_raw).glob(f"*/*/{t1_id}"))
    if t1_src_folders:
        t1_stem = f"{sub_id}_{ses_id}_T1w"
        t1_success, t1_status = convert_dicom_to_bids(
            t1_src_folders[0], anat_dir, t1_stem, is_bold=False
        )
    else:
        t1_success, t1_status = False, "missing_source_folder"
        missing_smri += 1

    manifest.append(
        {
            "Subject": subject_raw,
            "BIDS_Subject": sub_id,
            "BIDS_Session": ses_id,
            "fMRI_Image_ID": fmri_id,
            "fMRI_Acq_Date": fmri_date,
            "fMRI_Converted": fmri_success,
            "fMRI_Status": fmri_status,
            "T1_Image_ID": t1_id,
            "T1_Study_Date": t1_date,
            "T1_Converted": t1_success,
            "T1_Status": t1_status,
        }
    )

    if (idx + 1) % 100 == 0 or (idx + 1) == total:
        print(f"[{idx + 1}/{total}] processed (latest: {sub_id} {ses_id})")

In [ ]:
# ==========================================
# 4. SAVE MANIFEST & REPORT
# ==========================================
manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(MANIFEST_CSV, index=False)

print("\n--- BIDS Conversion Complete ---")
print(f"Total sessions processed:        {len(master_df)}")
print(f"Successfully converted fMRI:     {manifest_df['fMRI_Converted'].sum()}")
print(f"Successfully converted T1w:      {manifest_df['T1_Converted'].sum()}")
print(f"Missing fMRI folders on disk:    {missing_fmri}")
print(f"Missing T1w folders on disk:     {missing_smri}")
print(f"Manifest saved to:               {MANIFEST_CSV}")